# 실습 7. Kaplan-Meier와 로그순위법

사회약학 연구방법 노트의 '실습 7. Kaplan-Meier와 로그순위법'에 딸린 노트북입니다. NCCTG 폐암 자료(R survival 패키지의 cancer, 진행성 폐암 환자 228명)로 사건 변수와 중앙 추적기간을 만들고, Kaplan-Meier 생존곡선과 중앙생존기간, 제한평균생존시간을 구한 뒤 로그순위 검정으로 성별과 ECOG 수행능력에 따른 생존을 비교합니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 7에 있습니다.


## a. 실습 데이터 준비

Colab에서는 런타임이 새로 배정될 때마다 lifelines를 다시 설치합니다. 자료는 인터넷에서 읽어 오므로 인터넷 연결이 필요합니다.

### 셀 1. lifelines 설치 (Colab은 세션마다, 내 컴퓨터는 한 번)

In [ ]:
!pip install lifelines

### 셀 2. 패키지 불러오기와 버전 확인

In [ ]:
import numpy as np
import pandas as pd
import scipy, matplotlib, lifelines

print("pandas    ", pd.__version__)
print("numpy     ", np.__version__)
print("scipy     ", scipy.__version__)
print("matplotlib", matplotlib.__version__)
print("lifelines ", lifelines.__version__)

### 셀 3. NCCTG 폐암 자료 불러오기

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/"
       "csv/survival/cancer.csv")
lung = pd.read_csv(url)
print(lung.shape)
lung.head()

### 셀 4. status 확인과 사건 변수 만들기

In [ ]:
print(lung["status"].value_counts())

# 사건(사망) = 1, 중도절단 = 0 으로 새 열을 만듦
lung["event"] = (lung["status"] == 2).astype(int)
pd.crosstab(lung["status"], lung["event"])

### 셀 5. 사건 부호화를 잘못하면 (오류 없이 틀린 결과)

In [ ]:
from lifelines import KaplanMeierFitter

wrong = KaplanMeierFitter().fit(lung["time"], lung["status"])
right = KaplanMeierFitter().fit(lung["time"], lung["event"])
print("status 그대로:", wrong.event_table["observed"].sum(),
      "건, 중앙값", wrong.median_survival_time_, "일")
print("event 사용  :", right.event_table["observed"].sum(),
      "건, 중앙값", right.median_survival_time_, "일")

### 셀 6. 시간 단위 바꾸기와 사건 수 세기

In [ ]:
lung["months"] = lung["time"] / 30.4375   # 365.25일 / 12

n, d = len(lung), lung["event"].sum()
print("환자", n, "명, 사망", d, "명, 중도절단", n - d, "명")
print("사망 비율(추적 기간 무시):", round(d / n * 100, 1), "%")

py = lung["months"].sum() / 12            # 총 추적 인년
print("총 추적:", round(py, 1), "인년")
print("사망률:", round(d / py * 100, 1), "/ 100인년")

### 셀 7. 중앙 추적기간 (역 Kaplan-Meier)

In [ ]:
from lifelines.utils import median_survival_times, qth_survival_times

print("전체 time의 중앙값:", round(lung["months"].median(), 1))
alive = lung["event"] == 0
print("생존자 time의 중앙값:", round(lung.loc[alive, "months"].median(), 1))

# 역 Kaplan-Meier: 중도절단(1 - event)을 '사건'으로 놓는다
fu = KaplanMeierFitter().fit(lung["months"], 1 - lung["event"])
print("역 KM 중앙 추적기간:", round(fu.median_survival_time_, 1))
print(qth_survival_times([0.75, 0.25], fu.survival_function_))

## b. Kaplan-Meier 생존곡선

### 셀 8. Kaplan-Meier 추정

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(lung["months"], event_observed=lung["event"],
        label="All patients")
print(kmf)
kmf.survival_function_.head()

### 셀 9. 사건 표 (event_table)

In [ ]:
kmf.event_table.head(6)

### 셀 10. 처음 두 계단을 손으로 계산

In [ ]:
s1 = (1 - 1/228)                  # 0.16개월: 228명 중 1명 사망
s2 = s1 * (1 - 3/227)             # 0.36개월: 227명 중 3명 사망
print(round(s1, 6), round(s2, 6))

### 셀 11. 중앙생존기간과 95% 신뢰구간

In [ ]:
print("중앙생존기간:", round(kmf.median_survival_time_, 2), "개월")
median_survival_times(kmf.confidence_interval_)

### 셀 12. 시점별 생존율과 95% 신뢰구간

In [ ]:
ci = kmf.confidence_interval_
for t in [6, 12, 24]:
    s = kmf.survival_function_at_times(t).iloc[0]
    lo, hi = ci.loc[:t].iloc[-1]          # t 이전 마지막 계단
    print(f"{t:>2}개월 생존율 {s:.3f} (95% CI {lo:.3f}-{hi:.3f})")

### 셀 13. Greenwood 공식과 log-log 신뢰구간 직접 확인

In [ ]:
et = kmf.event_table.loc[:12]             # 12개월까지의 사건 표
d, n = et["observed"], et["at_risk"]
gw = (d / (n * (n - d))).sum()            # Greenwood 합
s = kmf.survival_function_at_times(12).iloc[0]
print("표준오차:", round(s * np.sqrt(gw), 4))

sigma = np.sqrt(gw) / abs(np.log(s))      # log-log 척도의 표준오차
print("log-log CI:", round(s ** np.exp(1.96 * sigma), 3),
      round(s ** np.exp(-1.96 * sigma), 3))

### 셀 14. Kaplan-Meier 곡선과 위험집합 표

In [ ]:
import matplotlib.pyplot as plt
from lifelines.plotting import add_at_risk_counts

fig, ax = plt.subplots(figsize=(7, 4))
kmf.plot_survival_function(ax=ax, show_censors=True)
ax.set_xlabel("Months")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1)
ax.set_xticks(range(0, 37, 6))
add_at_risk_counts(kmf, ax=ax)
plt.tight_layout()

### 셀 15. 성별 Kaplan-Meier 곡선

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
km_sex = {}
for code, name in [(1, "Male"), (2, "Female")]:
    g = lung[lung["sex"] == code]
    km_sex[name] = KaplanMeierFitter().fit(
        g["months"], g["event"], label=name)
    km_sex[name].plot_survival_function(ax=ax, show_censors=True)
ax.set_xlabel("Months")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1)
ax.set_xticks(range(0, 37, 6))
add_at_risk_counts(*km_sex.values(), ax=ax, rows_to_show=["At risk"])
plt.tight_layout()

### 셀 16. 성별 요약표 (논문 표 형식)

In [ ]:
rows = []
for name, k in km_sex.items():
    med = k.median_survival_time_
    lo, hi = median_survival_times(k.confidence_interval_).iloc[0]
    s12 = k.survival_function_at_times(12).iloc[0] * 100
    l12, h12 = k.confidence_interval_.loc[:12].iloc[-1] * 100
    rows.append({"group": name,
                 "n": k.event_table["at_risk"].iloc[0],
                 "deaths": k.event_table["observed"].sum(),
                 "median (95% CI)": f"{med:.1f} ({lo:.1f}-{hi:.1f})",
                 "1-yr % (95% CI)": f"{s12:.1f} ({l12:.1f}-{h12:.1f})"})
pd.DataFrame(rows)

### 셀 17. 24개월 제한평균생존시간(RMST)과 차이

In [ ]:
from lifelines.utils import restricted_mean_survival_time as rmst

def rmst_se(k, tau):
    """RMST(tau)와 그 표준오차 (Greenwood형 분산)"""
    et = k.event_table.loc[:tau]
    et = et[et["observed"] > 0]           # 사망이 있는 시점만
    area = np.array([rmst(k, t=tau) - rmst(k, t=t) for t in et.index])
    n, d = et["at_risk"], et["observed"]
    var = (area**2 * d / (n * (n - d))).sum()
    return rmst(k, t=tau), np.sqrt(var)

(r_m, se_m), (r_f, se_f) = [rmst_se(k, 24) for k in km_sex.values()]
diff, se = r_f - r_m, np.sqrt(se_m**2 + se_f**2)
print(f"24개월 RMST: 남 {r_m:.2f}, 여 {r_f:.2f}개월")
print(f"차이 {diff:.2f}개월 (95% CI {diff - 1.96*se:.2f}"
      f" to {diff + 1.96*se:.2f})")

## c. 로그순위법

### 셀 18. 로그순위 검정 (남 대 여)

In [ ]:
from lifelines.statistics import logrank_test

m = lung["sex"] == 1                       # 남성이면 True
res = logrank_test(lung.loc[m, "months"], lung.loc[~m, "months"],
                   event_observed_A=lung.loc[m, "event"],
                   event_observed_B=lung.loc[~m, "event"])
res.print_summary()
print(res.test_statistic, res.p_value)

### 셀 19. O, E, V를 직접 계산해 확인

In [ ]:
rows = []
for t in np.sort(lung.loc[lung["event"] == 1, "months"].unique()):
    r = lung[lung["months"] >= t]             # t 직전 위험집합
    dead = (r["months"] == t) & (r["event"] == 1)
    male = r["sex"] == 1
    rows.append({"n": len(r), "nA": male.sum(),
                 "d": dead.sum(), "dA": (dead & male).sum()})
tab = pd.DataFrame(rows)
E = (tab["d"] * tab["nA"] / tab["n"]).sum()
V = (tab["d"] * (tab["nA"] / tab["n"]) * (1 - tab["nA"] / tab["n"])
     * (tab["n"] - tab["d"]) / (tab["n"] - 1).clip(lower=1)).sum()
O = tab["dA"].sum()
print(f"남성 O = {O}, E = {E:.2f}, O-E = {O - E:.2f}, V = {V:.2f}")
print(f"chi2 = {(O - E)**2 / V:.2f}")

### 셀 20. 세 군 비교 (ECOG 0, 1, 2+)

In [ ]:
from lifelines.statistics import multivariate_logrank_test

e = lung.dropna(subset=["ph.ecog"]).copy()      # 결측 1명 제외
e["ecog"] = e["ph.ecog"].clip(upper=2).astype(int)   # 3 -> 2 ("2+")
print(e["ecog"].value_counts().sort_index())

res3 = multivariate_logrank_test(e["months"], e["ecog"], e["event"])
res3.print_summary()

### 셀 21. ECOG 군별 곡선

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
km_ecog = []
for g in [0, 1, 2]:
    s = e[e["ecog"] == g]
    lab = f"ECOG {g}" + ("+" if g == 2 else "")
    k = KaplanMeierFitter().fit(s["months"], s["event"], label=lab)
    k.plot_survival_function(ax=ax, ci_show=False, show_censors=True)
    km_ecog.append(k)
    print(lab, "median", round(k.median_survival_time_, 1))
ax.set_xlabel("Months")
ax.set_ylabel("Survival probability")
ax.set_xticks(range(0, 37, 6))
add_at_risk_counts(*km_ecog, ax=ax, rows_to_show=["At risk"])
plt.tight_layout()

### 셀 22. 두 군씩 비교와 다중비교 보정

In [ ]:
from lifelines.statistics import pairwise_logrank_test

pw = pairwise_logrank_test(e["months"], e["ecog"], e["event"])
out = pw.summary[["test_statistic", "p"]].copy()
out["p_bonferroni"] = (out["p"] * 3).clip(upper=1)   # 비교 3번
out.round(4)

### 셀 23. 가중 로그순위 검정 비교

In [ ]:
tests = {"log-rank": {},
         "Gehan-Breslow": {"weightings": "wilcoxon"},
         "Tarone-Ware": {"weightings": "tarone-ware"},
         "Peto-Peto": {"weightings": "peto"},
         "Fleming-Harrington(0,1)":
             {"weightings": "fleming-harrington", "p": 0, "q": 1}}
for name, kw in tests.items():
    r = logrank_test(lung.loc[m, "months"], lung.loc[~m, "months"],
                     lung.loc[m, "event"], lung.loc[~m, "event"],
                     **kw)
    print(f"{name:<24} chi2 = {r.test_statistic:6.2f}"
          f"  P = {r.p_value:.4f}")

## d. 과제 정답

**과제 1.** 셀 16의 성별 요약표에서 남성 행을 논문의 Results처럼 한 문장으로 옮기고, 이 행이 말해 주지 않는 것을 두 가지 이상 쓰세요. 성별 사망 비율, 100인년당 사망률, 평균 나이를 구해 비교하세요.

### 셀 24. 과제 1 정답. 성별 요약표의 한 행 읽기

In [ ]:
g = lung.groupby("sex")                    # 1 = 남성, 2 = 여성
hw1 = pd.DataFrame({"n": g.size(),
                    "deaths": g["event"].sum(),
                    "person_years": g["months"].sum() / 12})
hw1["deaths_%"] = hw1["deaths"] / hw1["n"] * 100
hw1["rate_per_100py"] = hw1["deaths"] / hw1["person_years"] * 100
hw1["mean_age"] = g["age"].mean()
hw1.round(1)

**과제 2.** 셀 18의 성별 로그순위 검정을 ECOG 수행능력(0, 1, 2+, 셀 20의 e)으로 층화해 다시 하세요. 층마다 셀 19처럼 O, E, V를 구해 더합니다. 층화하지 않은 결과와 비교해 χ²가 왜 이만큼 달라지는지(또는 왜 거의 그대로인지) 설명하세요.

### 셀 25. 과제 2 정답. ECOG로 층화한 로그순위 검정

In [ ]:
from scipy import stats

def oev(df):
    """남성(sex == 1)의 관찰 사망 O, 기대 사망 E, 분산 V"""
    O = E = V = 0
    for t in np.sort(df.loc[df["event"] == 1, "months"].unique()):
        r = df[df["months"] >= t]                    # t 직전 위험집합
        dead = (r["months"] == t) & (r["event"] == 1)
        male = r["sex"] == 1
        n, nA, d = len(r), male.sum(), dead.sum()
        O += (dead & male).sum()
        E += d * nA / n
        V += d * (nA / n) * (1 - nA / n) * (n - d) / max(n - 1, 1)
    return O, E, V

print(pd.crosstab(e["sex"], e["ecog"], normalize="index").round(3))
strata = pd.DataFrame({g: oev(s) for g, s in e.groupby("ecog")},
                      index=["O", "E", "V"]).T      # ECOG 층마다 한 행
print(strata.assign(O_minus_E=strata["O"] - strata["E"]).round(2))
for name, (O, E, V) in [("stratified  ", strata.sum()),
                        ("unstratified", oev(e))]:
    chi2 = (O - E) ** 2 / V
    print(f"{name} O-E = {O - E:5.2f}, V = {V:5.2f}, "
          f"chi2 = {chi2:5.2f}, P = {stats.chi2.sf(chi2, 1):.4f}")

**과제 3.** 동료가 1년 생존율을 성별로 비교하려고 12개월 전에 중도절단된 환자를 빼고, 남은 환자를 '12개월 안에 사망 / 12개월 넘게 생존'으로 나눠 카이제곱 검정을 한 뒤 "1년 생존율은 남성 29.2%, 여성 45.5%였다(P = 0.039)"라고 썼습니다. 동료의 분석을 재현하고, 틀린 곳 두 가지를 찾아 왜 틀렸는지 설명하세요.

### 셀 26. 과제 3 정답. 12개월 생존 여부를 카이제곱으로 비교한 분석

In [ ]:
from lifelines.statistics import survival_difference_at_fixed_point_in_time_test

known = (lung["months"] >= 12) | (lung["event"] == 1)  # 1년 생존 여부를 아는 환자
print("excluded:", (~known).sum(), "(male, female) =",
      lung.loc[~known, "sex"].value_counts().sort_index().tolist())
d12 = lung[known].assign(dead12=lambda x: (x["months"] < 12).astype(int))
tab12 = pd.crosstab(d12["sex"], d12["dead12"])        # 열 1 = 12개월 안 사망
print(tab12)
print("colleague 1-yr survival:", (tab12[0] / tab12.sum(axis=1)).round(3).tolist(),
      " chi-square P =", round(stats.chi2_contingency(tab12)[1], 3))
fp = survival_difference_at_fixed_point_in_time_test(
    12, km_sex["Male"], km_sex["Female"])
print("Kaplan-Meier 1-yr survival:",
      [round(float(k.predict(12)), 3) for k in km_sex.values()],
      " P =", round(fp.p_value, 3))